# AlexandriaX Subtask 1: Context-Aware English-to-Dialectal Arabic Dialogue Translation Baseline

This notebook represents a baseline for English-to-dialectal-Arabic conversation translation using the Alexandria dataset. It trains on the Hugging Face `UBC-NLP/alexandria` training split, evaluates on the local `development_data` split, and creates a CodaBench-shaped submission zip for that development split.

The local development data is intentionally used for generation-based evaluation and submission file creation.

You can use the same notebook during the test phase; just replace development_data with test_data.

## 1. Setup and configuration

This section defines paths, split names, and run switches.

In [ ]:
from __future__ import annotations

import gc
import json
import zipfile
from collections import defaultdict
from pathlib import Path

import torch
from datasets import Dataset, get_dataset_config_names, get_dataset_split_names, load_dataset
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

DATASET_NAME = "UBC-NLP/alexandria"
COUNTRIES = ["EG", "JO", "LB", "LY", "MA", "MR", "OM", "PS", "SA", "SD", "SY", "TN", "YE"]

DEVELOPMENT_DATA_DIR = Path("development_data")
DEVELOPMENT_REFERENCE_PATH = DEVELOPMENT_DATA_DIR / "reference_data" / "references.jsonl"

OUTPUT_ROOT = Path("outputs")
CHECKPOINT_DIR = OUTPUT_ROOT / "nilechat_alexandriaX_lora"
EVAL_DIR = OUTPUT_ROOT / "evaluation"
SUBMISSION_DIR = OUTPUT_ROOT / "submissions"
SCORE_DIR = OUTPUT_ROOT / "scores"

RUN_DEVELOPMENT_EVALUATION = True
RUN_DEVELOPMENT_SUBMISSION = True

for directory in (OUTPUT_ROOT, EVAL_DIR, SUBMISSION_DIR, SCORE_DIR):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())
else:
    print("No GPU found. Training NileChat-3B on CPU is not practical.")


CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
BF16 supported: True


## 2. Dataset loading

Training examples come from the Hugging Face training split. Development examples come from local JSONL files under `development_data`, with references loaded from `development_data/reference_data/references.jsonl`. (downloaded from CodaBench competition).


In [ ]:
def discover_hf_splits(dataset_name: str, countries: list[str]) -> dict[str, set[str]]:
    available_configs = set(get_dataset_config_names(dataset_name))
    split_map: dict[str, set[str]] = {}

    for country in countries:
        if country not in available_configs:
            split_map[country] = set()
            continue
        split_map[country] = set(get_dataset_split_names(dataset_name, country))

    return split_map


hf_split_map = discover_hf_splits(DATASET_NAME, COUNTRIES)
TRAIN_COUNTRIES = [country for country in COUNTRIES if "train" in hf_split_map.get(country, set())]

print("Hugging Face train countries:", TRAIN_COUNTRIES)
print("Countries without HF train split:", [c for c in COUNTRIES if c not in TRAIN_COUNTRIES])


Hugging Face train countries: ['EG', 'JO', 'LB', 'MA', 'MR', 'OM', 'PS', 'SA', 'SY', 'TN', 'YE']
Countries without HF train split: ['LY', 'SD']


In [ ]:
def _turn_order(turn: dict, fallback: int = 0) -> int:
    try:
        return int(turn.get("turn_order", fallback))
    except (TypeError, ValueError):
        return fallback


def sorted_turns(turns: list[dict]) -> list[dict]:
    return sorted(turns or [], key=lambda turn: _turn_order(turn))


def read_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def normalize_hf_record(row: dict) -> dict:
    english_turns = sorted_turns(row.get("english_conversation", []))
    dialect_turns = {
        _turn_order(turn, index + 1): turn
        for index, turn in enumerate(sorted_turns(row.get("dialectal_conversation", [])))
    }

    turns = []
    whole_conversation_lines = []
    for index, english_turn in enumerate(english_turns, start=1):
        order = _turn_order(english_turn, index)
        reference_turn = dialect_turns.get(order, {})
        speaker = str(english_turn.get("speaker", "")).strip()
        sentence = str(english_turn.get("text", "")).strip()
        whole_conversation_lines.append(f"{speaker}: {sentence}" if speaker else sentence)
        turns.append({
            "turn_order": order,
            "speaker": speaker,
            "sentence": sentence,
            "direction": str(english_turn.get("direction", "")).strip(),
            "reference": str(reference_turn.get("text", "")).strip(),
        })

    return {
        "conv_id": str(row.get("conv_id", "")).strip(),
        "country": str(row.get("country", "")).strip(),
        "domain": str(row.get("domain", "")).strip(),
        "dialect": str(row.get("dialect", "Arabic Dialect")).strip() or "Arabic Dialect",
        "participants": str(row.get("participants", "")).strip(),
        "whole_conversation": "\n\n".join(whole_conversation_lines),
        "turns": turns,
    }


def normalize_development_input_record(row: dict) -> dict:
    turns = []
    for index, turn in enumerate(sorted_turns(row.get("turns", [])), start=1):
        turns.append({
            "turn_order": _turn_order(turn, index),
            "speaker": str(turn.get("speaker", "")).strip(),
            "sentence": str(turn.get("sentence", turn.get("text", ""))).strip(),
            "direction": str(turn.get("direction", "")).strip(),
            "reference": "",
        })

    return {
        "conv_id": str(row.get("conv_id", "")).strip(),
        "country": str(row.get("country", "")).strip(),
        "domain": str(row.get("domain", "")).strip(),
        "dialect": str(row.get("dialect", "Arabic Dialect")).strip() or "Arabic Dialect",
        "participants": str(row.get("participants", "")).strip(),
        "whole_conversation": str(row.get("whole_conversation", "")).strip(),
        "turns": turns,
    }


def load_reference_lookup(path: Path) -> dict[tuple[str, str, int], str]:
    references = {}
    for record in read_jsonl(path):
        country = str(record.get("country", "")).strip()
        conv_id = str(record.get("conv_id", "")).strip()
        for index, turn in enumerate(sorted_turns(record.get("turns", [])), start=1):
            turn_order = _turn_order(turn, index)
            key = (country, conv_id, turn_order)
            if key in references:
                raise ValueError(f"Duplicate reference key: {key}")
            references[key] = str(turn.get("reference", "")).strip()
    return references


def attach_references(records: list[dict], references: dict[tuple[str, str, int], str]) -> list[dict]:
    missing = []
    for record in records:
        country = record["country"]
        conv_id = record["conv_id"]
        for turn in record.get("turns", []):
            key = (country, conv_id, int(turn["turn_order"]))
            reference = references.get(key)
            if reference is None:
                missing.append(key)
                continue
            turn["reference"] = reference

    if missing:
        raise ValueError(f"Missing {len(missing)} development references; first missing keys: {missing[:5]}")
    return records


def format_history(history: list[dict]) -> str:
    if not history:
        return "No previous turns (Start of conversation)."

    lines = []
    for item in history:
        speaker = item.get("speaker", "Speaker") or "Speaker"
        lines.append(f"{speaker}: {item['sentence']}\nTranslation: {item['translation']}")
    return "\n".join(lines)


def build_prompt(record: dict, turn: dict, history: list[dict]) -> str:
    dialect = record.get("dialect") or "Arabic Dialect"
    domain = record.get("domain") or "Unknown Domain"
    participants = record.get("participants") or "Unknown Participants"
    direction = turn.get("direction") or "Unknown"
    speaker = turn.get("speaker") or "Unknown Speaker"

    return (
        f"You are an expert translator. Translate the English sentence into {dialect}.\n\n"
        f"### Metadata:\n"
        f"- Country: {record.get('country', '')}\n"
        f"- Domain: {domain}\n"
        f"- Participants: {participants}\n"
        f"- Speaker: {speaker}\n"
        f"- Speaker Direction: {direction}\n\n"
        f"### Conversation History:\n"
        f"{format_history(history)}\n\n"
        f"### Sentence to Translate:\n"
        f"{turn.get('sentence', '').strip()}\n\n"
        f"### Translation:\n"
    )


def create_finetuning_pairs(record: dict) -> list[dict]:
    pairs = []
    history: list[dict] = []

    for turn in sorted_turns(record.get("turns", [])):
        sentence = turn.get("sentence", "").strip()
        reference = turn.get("reference", "").strip()
        if not sentence or not reference:
            continue

        pairs.append({
            "prompt": build_prompt(record, turn, history),
            "response": reference,
            "country": record.get("country", ""),
            "conv_id": record.get("conv_id", ""),
            "turn_order": turn.get("turn_order"),
        })
        history.append({
            "speaker": turn.get("speaker", ""),
            "sentence": sentence,
            "translation": reference,
        })

    return pairs


In [ ]:
def load_hf_train_records(countries: list[str]) -> list[dict]:
    records: list[dict] = []

    for country in tqdm(countries, desc="Loading HF train countries", unit="country"):
        dataset = load_dataset(DATASET_NAME, country, split="train")
        country_records = [normalize_hf_record(row) for row in dataset]
        records.extend(country_records)
        print(f"Loaded {len(country_records):>5} train conversations for {country}")

    return records


def load_development_records(data_dir: Path, reference_path: Path) -> list[dict]:
    if not data_dir.exists():
        raise FileNotFoundError(f"Development data directory does not exist: {data_dir}")
    if not reference_path.exists():
        raise FileNotFoundError(f"Development reference file does not exist: {reference_path}")

    input_paths = sorted(data_dir.glob("alexandria_*_dev_input.jsonl"))
    if not input_paths:
        raise FileNotFoundError(f"No development input files found under {data_dir}")

    records: list[dict] = []
    for path in tqdm(input_paths, desc="Loading development input files", unit="file"):
        country_records = [normalize_development_input_record(row) for row in read_jsonl(path)]
        records.extend(country_records)
        print(f"Loaded {len(country_records):>5} development conversations from {path.name}")

    references = load_reference_lookup(reference_path)
    return attach_references(records, references)


hf_train_records = load_hf_train_records(TRAIN_COUNTRIES)
development_records = load_development_records(DEVELOPMENT_DATA_DIR, DEVELOPMENT_REFERENCE_PATH)

train_pairs = [pair for record in hf_train_records for pair in create_finetuning_pairs(record)]
development_pairs = [pair for record in development_records for pair in create_finetuning_pairs(record)]

print("Train conversations:", len(hf_train_records))
print("Development conversations:", len(development_records))
print("Train turns:", len(train_pairs))
print("Development turns:", len(development_pairs))


Loading HF train countries:   9%|▉         | 1/11 [00:00<00:08,  1.13country/s]

Loaded   982 train conversations for EG


Loading HF train countries:  18%|█▊        | 2/11 [00:01<00:07,  1.20country/s]

Loaded  1730 train conversations for JO


Loading HF train countries:  27%|██▋       | 3/11 [00:02<00:07,  1.13country/s]

...

Loading HF train countries: 100%|██████████| 11/11 [00:10<00:00,  1.06country/s]


Loaded   988 train conversations for YE


Loading development input files: 100%|██████████| 11/11 [00:00<00:00, 175.78file/s]

Loaded   352 development conversations from alexandria_EG_dev_input.jsonl
Loaded   352 development conversations from alexandria_JO_dev_input.jsonl
Loaded   371 development conversations from alexandria_LB_dev_input.jsonl
Loaded   354 development conversations from alexandria_MA_dev_input.jsonl
Loaded   352 development conversations from alexandria_MR_dev_input.jsonl
Loaded   381 development conversations from alexandria_OM_dev_input.jsonl
Loaded   352 development conversations from alexandria_PS_dev_input.jsonl
Loaded   363 development conversations from alexandria_SA_dev_input.jsonl
Loaded   347 development conversations from alexandria_SY_dev_input.jsonl
Loaded   378 development conversations from alexandria_TN_dev_input.jsonl
Loaded   361 development conversations from alexandria_YE_dev_input.jsonl


Train conversations: 21146
Development conversations: 3963
Train turns: 66480
Development turns: 12250


## 3. Training

This baseline trains a LoRA adapter for NileChat-3B-Base on all available HF `train` conversations. During training, validation loss is computed on the local development set built from `development_data`.


In [ ]:
MODEL_NAME = "UBC-NLP/NileChat-3B-Base"

MAX_SEQ_LENGTH = 1024
NUM_EPOCHS = 1
BATCH_SIZE = 32  # tuned for one A100 80GB in the original baseline
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4

USE_4BIT = True
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05


GENERATION_BATCH_SIZE = 256
MAX_NEW_TOKENS = 128


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("EOS token:", tokenizer.eos_token)
print("PAD token:", tokenizer.pad_token)


EOS token: <|endoftext|>
PAD token: <|endoftext|>


In [ ]:
def build_sft_dataset(pairs: list[dict]) -> Dataset:
    rows = []
    for item in pairs:
        rows.append({
            "prompt": item["prompt"],
            "completion": item["response"].strip() + tokenizer.eos_token,
        })
    return Dataset.from_list(rows)


def maybe_sample_dataset(dataset: Dataset, sample_size: int | None, seed: int = 42) -> Dataset:
    if sample_size is None or sample_size >= len(dataset):
        return dataset
    return dataset.shuffle(seed=seed).select(range(sample_size))


train_dataset = build_sft_dataset(train_pairs)
eval_dataset = build_sft_dataset(development_pairs)

print(train_dataset)
print(eval_dataset)
print(train_dataset[0])


Dataset({
    features: ['prompt', 'completion'],
    num_rows: 66480
})
Dataset({
    features: ['prompt', 'completion'],
    num_rows: 12250
})
{'prompt': "You are an expert translator. Translate the English sentence into Egyptian Arabic (Cairene) Dialect.\n\n### Metadata:\n- Country: EG\n- Domain: Agriculture and farming\n- Participants: Wholesale Buyer, Wholesale Seller\n- Speaker: Wholesale Buyer\n- Speaker Direction: male -> female\n\n### Conversation History:\nNo previous turns (Start of conversation).\n\n### Sentence to Translate:\nGood morning. I'm looking to source 10 tons of premium artichokes for export. People say the best quality in Obour comes from your section, is that right?\n\n### Translation:\n", 'completion': 'صباح الخير، عايز عشرة طن من الخرشوف الكويس للتصدير، بيقولوا ان احسن جودة في سوق العبور بتيجي من عندكم، صحيح؟<|endoftext|>'}


In [ ]:
def model_compute_dtype() -> torch.dtype:
    return torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16


def build_quantization_config() -> BitsAndBytesConfig | None:
    if not USE_4BIT:
        return None

    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=model_compute_dtype(),
        bnb_4bit_use_double_quant=True,
    )


bf16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
compute_dtype = model_compute_dtype()
quantization_config = build_quantization_config()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=compute_dtype,
    quantization_config=quantization_config,
    trust_remote_code=True,
)

model.config.use_cache = False

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

print("Model loaded for training.")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/435 [00:00<?, ?it/s]

/mnt/home/aelmekki/vllm_serve/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model loaded for training.


In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

training_args = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    max_length=MAX_SEQ_LENGTH,
    completion_only_loss=True,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=200,
    bf16=bf16_supported,
    fp16=not bf16_supported,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch",
    report_to="none",
    remove_unused_columns=True,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/66480 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/66480 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/12250 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/12250 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

trainer.model.save_pretrained(CHECKPOINT_DIR)
tokenizer.save_pretrained(CHECKPOINT_DIR)

print(f"Saved LoRA adapter and tokenizer to: {CHECKPOINT_DIR}")

# Free training objects before reloading the saved checkpoint for inference.
del trainer
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
200,1.358823,1.400096
400,1.195924,1.329403
520,1.218537,1.325902


Saved LoRA adapter and tokenizer to: outputs/nilechat_alexandriaX_lora


## 4. Load the saved checkpoint for inference

Use the trained model for inference by laoding it from the existing checkpoint.


In [ ]:
adapter_config_path = CHECKPOINT_DIR / "adapter_config.json"
if not adapter_config_path.exists():
    raise FileNotFoundError(
        f"No LoRA adapter found at {CHECKPOINT_DIR}. Run the training section first, "
        "or set CHECKPOINT_DIR to a completed checkpoint."
    )

tokenizer_source = CHECKPOINT_DIR if (CHECKPOINT_DIR / "tokenizer_config.json").exists() else MODEL_NAME
tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_source,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=model_compute_dtype(),
    quantization_config=build_quantization_config(),
    trust_remote_code=True,
)

fine_tuned_model = PeftModel.from_pretrained(base_model, CHECKPOINT_DIR)
if hasattr(fine_tuned_model, "gradient_checkpointing_disable"):
    fine_tuned_model.gradient_checkpointing_disable()
fine_tuned_model.eval()
fine_tuned_model.config.use_cache = True

print(f"Loaded fine-tuned model from: {CHECKPOINT_DIR}")


Loading weights:   0%|          | 0/435 [00:00<?, ?it/s]

/mnt/home/aelmekki/vllm_serve/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Loaded fine-tuned model from: outputs/nilechat_alexandriaX_lora


## 5. Prediction and scoring helpers

This prediction helper translates one turn position at a time across all conversations. That lets later turns use generated translations from earlier turns as conversation history (context-aware MT).


In [ ]:
def count_turns(records: list[dict]) -> int:
    return sum(len(record.get("turns", [])) for record in records)


def chunks(items: list, batch_size: int):
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]


def clean_generation(text: str) -> str:
    text = text.strip()
    for marker in ("\n###", "### Sentence to Translate:", "### Translation:"):
        if marker in text:
            text = text.split(marker, 1)[0].strip()
    return text


def generate_translations(prompts: list[str], model, tokenizer, max_new_tokens: int = MAX_NEW_TOKENS) -> list[str]:
    previous_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated_tokens = outputs[:, inputs["input_ids"].shape[1]:]
    decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    tokenizer.padding_side = previous_padding_side
    return [clean_generation(text) for text in decoded]


def generate_prediction_records(
    records: list[dict],
    model,
    tokenizer,
    batch_size: int = GENERATION_BATCH_SIZE,
    desc: str = "Generating predictions",
) -> list[dict]:
    histories: dict[int, list[dict]] = defaultdict(list)
    outputs = [
        {"conv_id": record["conv_id"], "country": record["country"], "turns": []}
        for record in records
    ]
    max_turns = max((len(record.get("turns", [])) for record in records), default=0)

    with tqdm(total=count_turns(records), desc=desc, unit="turn") as progress:
        for turn_position in range(max_turns):
            active = []
            for record_index, record in enumerate(records):
                turns = sorted_turns(record.get("turns", []))
                if turn_position >= len(turns):
                    continue
                turn = turns[turn_position]
                prompt = build_prompt(record, turn, histories[record_index])
                active.append((record_index, turn, prompt))

            for batch in chunks(active, batch_size):
                prompts = [item[2] for item in batch]
                translations = generate_translations(prompts, model, tokenizer)
                for (record_index, turn, _), translation in zip(batch, translations):
                    outputs[record_index]["turns"].append({
                        "turn_order": int(turn["turn_order"]),
                        "prediction": translation,
                    })
                    histories[record_index].append({
                        "speaker": turn.get("speaker", ""),
                        "sentence": turn.get("sentence", ""),
                        "translation": translation,
                    })
                progress.update(len(batch))

    return outputs


def write_jsonl(records: list[dict], path: Path, desc: str | None = None) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    iterator = tqdm(records, desc=desc or f"Writing {path.name}", unit="conversation")
    with path.open("w", encoding="utf-8") as handle:
        for record in iterator:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
    return path


def make_submission_zip(predictions_jsonl: Path, zip_path: Path) -> Path:
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(predictions_jsonl, arcname="predictions.jsonl")
    print(f"Wrote {zip_path}")
    return zip_path


In [ ]:
def prediction_map(prediction_records: list[dict]) -> dict[tuple[str, str, int], str]:
    mapped = {}
    for record in prediction_records:
        country = str(record.get("country", "")).strip()
        conv_id = str(record.get("conv_id", "")).strip()
        for index, turn in enumerate(record.get("turns", []), start=1):
            turn_order = int(turn.get("turn_order", index))
            key = (country, conv_id, turn_order)
            if key in mapped:
                raise ValueError(f"Duplicate prediction key: {key}")
            mapped[key] = str(turn.get("prediction", "")).strip()
    return mapped


def reference_map(reference_records: list[dict]) -> dict[tuple[str, str, int], str]:
    mapped = {}
    for record in reference_records:
        country = str(record.get("country", "")).strip()
        conv_id = str(record.get("conv_id", "")).strip()
        for index, turn in enumerate(sorted_turns(record.get("turns", [])), start=1):
            reference = str(turn.get("reference", "")).strip()
            if not reference:
                continue
            turn_order = int(turn.get("turn_order", index))
            key = (country, conv_id, turn_order)
            if key in mapped:
                raise ValueError(f"Duplicate reference key: {key}")
            mapped[key] = reference
    return mapped


def score_prediction_records(
    prediction_records: list[dict],
    reference_records: list[dict],
    output_path: Path | None = None,
    desc: str = "Scoring countries",
) -> dict:
    try:
        from sacrebleu.metrics import BLEU, CHRF
    except ImportError as exc:
        raise ImportError("Install scoring dependencies with: pip install sacrebleu sentencepiece") from exc

    predictions = prediction_map(prediction_records)
    references = reference_map(reference_records)

    missing = sorted(set(references) - set(predictions))
    extra = sorted(set(predictions) - set(references))
    if missing or extra:
        raise ValueError(f"Prediction/reference mismatch. Missing={missing[:5]}, extra={extra[:5]}")

    try:
        bleu = BLEU(tokenize="flores200", effective_order=False)
    except Exception as exc:
        raise RuntimeError(
            "Could not initialize SacreBLEU's flores200 tokenizer. "
            "Install/upgrade sacrebleu and sentencepiece."
        ) from exc
    chrf = CHRF(word_order=2)

    by_country: dict[str, list[tuple[str, str]]] = defaultdict(list)
    for key in tqdm(sorted(references), desc="Aligning predictions", unit="turn"):
        country = key[0]
        by_country[country].append((predictions[key], references[key]))

    scores: dict[str, float | int] = {}
    spbleu_values = []
    chrfpp_values = []
    for country, rows in tqdm(sorted(by_country.items()), desc=desc, unit="country"):
        hypotheses = [prediction for prediction, _ in rows]
        refs = [reference for _, reference in rows]
        spbleu = bleu.corpus_score(hypotheses, [refs]).score
        chrfpp = chrf.corpus_score(hypotheses, [refs]).score
        spbleu_values.append(spbleu)
        chrfpp_values.append(chrfpp)
        scores[f"spbleu_{country}"] = round(spbleu, 6)
        scores[f"chrfpp_{country}"] = round(chrfpp, 6)

    scores["spbleu_avg"] = round(sum(spbleu_values) / len(spbleu_values), 6)
    scores["chrfpp_avg"] = round(sum(chrfpp_values) / len(chrfpp_values), 6)
    scores["num_countries"] = len(by_country)
    scores["num_turns"] = len(references)
    scores["num_conversations"] = len({key[:2] for key in references})

    if output_path is not None:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        output_path.write_text(json.dumps(scores, ensure_ascii=False, indent=2, sort_keys=True), encoding="utf-8")
        print(f"Scores saved to {output_path}")

    return scores


def display_scores(scores: dict) -> None:
    summary_keys = ["spbleu_avg", "chrfpp_avg", "num_countries", "num_conversations", "num_turns"]
    country_rows = []
    for key, value in scores.items():
        if not key.startswith("spbleu_") or key == "spbleu_avg":
            continue
        country = key.removeprefix("spbleu_")
        country_rows.append({
            "country": country,
            "spbleu": value,
            "chrfpp": scores.get(f"chrfpp_{country}"),
        })

    try:
        import pandas as pd
        from IPython.display import display

        display(pd.DataFrame([{
            "spbleu_avg": scores["spbleu_avg"],
            "chrfpp_avg": scores["chrfpp_avg"],
            "countries": scores["num_countries"],
            "conversations": scores["num_conversations"],
            "turns": scores["num_turns"],
        }]))
        display(pd.DataFrame(country_rows).sort_values("country").reset_index(drop=True))
    except Exception:
        print(json.dumps({key: scores[key] for key in summary_keys}, indent=2, ensure_ascii=False))
        print(json.dumps(sorted(country_rows, key=lambda row: row["country"]), indent=2, ensure_ascii=False))


## 6. Evaluation on the local development split

This evaluates the saved checkpoint on `development_data`, writes raw development predictions, computes local scores using the development references, and displays the score summary directly in the notebook.


In [ ]:
if RUN_DEVELOPMENT_EVALUATION:
    development_predictions = generate_prediction_records(
        development_records,
        fine_tuned_model,
        tokenizer,
        desc="Generating development predictions",
    )
    development_predictions_path = write_jsonl(
        development_predictions,
        EVAL_DIR / "development_predictions.jsonl",
        desc="Writing development predictions",
    )
    development_scores = score_prediction_records(
        development_predictions,
        development_records,
        SCORE_DIR / "development_scores.json",
        desc="Scoring development countries",
    )
    display_scores(development_scores)
    development_scores


Scoring development countries: 100%|██████████| 11/11 [00:06<00:00,  1.82country/s]

Scores saved to outputs/scores/development_scores.json


,spbleu_avg,chrfpp_avg,countries,conversations,turns
0,21.822554,37.954745,11,3963,12250


,country,spbleu,chrfpp
0,EG,26.845631,41.447345
1,JO,27.811065,43.697727
2,LB,24.366649,39.913187
3,MA,14.606706,30.504620
4,MR,9.014372,26.246150
5,OM,20.491374,37.414216
6,PS,25.025101,41.026990
7,SA,25.545494,41.933916
8,SY,29.811550,45.957149
9,TN,19.633637,35.132828


## 7. Development submission file generation

This section creates a CodaBench zip submission file for the local development split. The zip contains one file named `predictions.jsonl`, where each line is one conversation with turn-level predictions.


In [ ]:
if RUN_DEVELOPMENT_SUBMISSION:
    if "development_predictions" not in globals():
        development_predictions = generate_prediction_records(
            development_records,
            fine_tuned_model,
            tokenizer,
            desc="Generating development submission predictions",
        )

    development_submission_path = write_jsonl(
        development_predictions,
        SUBMISSION_DIR / "development_predictions.jsonl",
        desc="Writing development submission predictions",
    )
    make_submission_zip(development_submission_path, SUBMISSION_DIR / "submission_development.zip")


Writing development submission predictions: 100%|██████████| 3963/3963 [00:00<00:00, 98383.71conversation/s]


Wrote outputs/submissions/submission_development.zip
